In [25]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import gensim
import random

In [6]:
df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [7]:
# Embedding
from gensim.models import Word2Vec

model_CBOW = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=10
)

model_S = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

In [8]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_CBOW.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['ants', 'leaf', 'acceptted', 'dissertation', 'centuries', 'ingratiating', 'obstacles', 'suspicion', 'desagreable', 'pre-set', 'calamity', 'veterinarian', 'thourgh', 'ideal', 'Hurt', 'Chirimba', 'whash', 'Unfair', 'Mugged', 'trunk', 'M.Phil', '2/3', 'student-like', 'prosperity', 'persoon', 'Mathematics', 'temporary', 'al', 'Begining', 'striven']


In [9]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_S.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['ants', 'leaf', 'acceptted', 'dissertation', 'centuries', 'ingratiating', 'obstacles', 'suspicion', 'desagreable', 'pre-set', 'calamity', 'veterinarian', 'thourgh', 'ideal', 'Hurt', 'Chirimba', 'whash', 'Unfair', 'Mugged', 'trunk', 'M.Phil', '2/3', 'student-like', 'prosperity', 'persoon', 'Mathematics', 'temporary', 'al', 'Begining', 'striven']


In [10]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_CBOW.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage CBOW: {token_coverage:.2f}%")

Token-level coverage CBOW: 97.18%


In [11]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_S.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage S: {token_coverage:.2f}%")

Token-level coverage S: 97.18%


In [12]:
print(model_CBOW.wv.most_similar("leukemia", topn=10))
print(model_S.wv.most_similar("leukemia", topn=10))

[('seriously', 0.9220069050788879), ('wristwatch', 0.9190288782119751), ('gradmother', 0.9137222766876221), ('80', 0.9126213788986206), ('grandfather', 0.910856306552887), ('cerebral', 0.9074204564094543), ('TE', 0.905414879322052), ('unexpectedly', 0.9045116901397705), ('stealing', 0.9031814336776733), ('beaten', 0.9022976160049438)]
[('gradmother', 0.9768155813217163), ('paternal', 0.958458423614502), ('collapsed', 0.9583346843719482), ('eleven', 0.9498734474182129), ('tragically', 0.9477477669715881), ('thirteen', 0.9396410584449768), ('lung', 0.9241835474967957), ('beloved', 0.9241205453872681), ('discharged', 0.9188069105148315), ('dying', 0.9184858202934265)]


In [13]:
print([word for word in model_vocab if len(word) <= 2])
print([word for word in dataset_vocab if len(word) <= 2])

['km', 'TE', "'m", '80', 'my', 'on', '``', 'in', 'No', '!', '20', 'he', '%', 'As', 'am', 'He', '9', "'", '[', ']', '4', 'e', 'of', 'OK', 'ca', 'we', 'pm', 'X', ')', '8', 'Ca', 'is', 'it', '(', '-', '$', 'B+', '11', 'an', 'do', 'Of', '16', 'if', 'My', '1', 'a', 'Ye', ';', '2A', 'as', 'to', '10', '5', 'up', 'Mr', 'so', '22', 'It', 'By', '12', "''", '3', 'An', '13', 'C', '35', '19', ':', '14', '60', 'Is', '.', 'go', ',', '84', "'s", 'NO', 'ex', 'i', 'aa', '18', 'or', 'B', 'at', '7', 'be', '30', 'me', 'To', 'us', 'by', '17', 'TV', '40', '&', 'So', 'PY', '15', '21', 'Oh', 'I.', 'If', 'ok', 'F4', '?', 'Km', 'A', 'LP', '/', '50', 'We', 'wo', 'TA', 'no', 'At', 'II', '6', 'C+', 'I', 'On', 'm', '2', "'d", 'In', 'D']
['km', 'al', 't', 'TE', "'m", 'M.', 'ne', 'WD', '80', 'Mt', '53', 'Do', 'my', 'on', '5B', 'Rs', '``', '31', '+2', 'in', 'No', '!', '20', 'El', '79', 'F', 'he', '%', 'm.', 'As', 'T', 'am', 'LA', 'He', '9', '95', "'", '[', ']', '4', 'e', 'of', 'E', 'OK', 'ca', 'we', 'hi', 'Kg', 'pm', '

In [24]:
# Pooling using mean
def mean_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.mean(vectors, axis=0)

# Pooling using max
def max_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.max(vectors, axis=0)

# Create a mean pooled vector for each document, for both embedding models
X_mean_cbow = np.vstack([
    mean_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_mean_skipgram = np.vstack([
    mean_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])



# Create a max pooled vector for each document, for both embedding models
X_max_cbow = np.vstack([
    max_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_max_skipgram = np.vstack([
    max_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

# Vector shapes should be the same
print(X_mean_cbow.shape)
print(X_mean_skipgram.shape)
print(X_max_cbow.shape)
print(X_max_skipgram.shape)


(7516, 100)
(7516, 100)
(7516, 100)
(7516, 100)


In [26]:
y = df["emotion"]

implementations = {
    "CBOW + mean": X_mean_cbow,
    "Skip-gram + mean": X_mean_skipgram,
    "CBOW + max": X_max_cbow,
    "Skip-gram + max": X_max_skipgram,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    results[name] = scores
    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

CBOW + mean: mean=0.3538, std=0.0114
Skip-gram + mean: mean=0.4287, std=0.0084
CBOW + max: mean=0.3314, std=0.0155
Skip-gram + max: mean=0.3833, std=0.0142
